# 9 · Distributed training with Ray Train

`harness/train.py`'s single-process loop is right for the tiny tier and wrong for
`bdd-full`. **Ray Train** runs the *same* training function across N data-parallel
workers. Below we run it locally with 2 CPU workers; on a cluster you change only
`num_workers` / `use_gpu`.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
try:
    import ray
    from ray.train import ScalingConfig
    from ray.train.torch import TorchTrainer
    HAVE_RAY = True
except Exception as e:
    HAVE_RAY = False; print("Ray not installed -> showing the code only:", e)

2026-05-25 16:13:38,698	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-25 16:13:38,987	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-05-25 16:13:40,594	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


### Wrap the per-worker training step and fit across 2 workers

In [3]:
def train_func(cfg):
    import numpy as np, torch, torch.nn.functional as F
    import ray.train
    from harness.data import load_split, class_names
    from harness.model import build_model
    rank = ray.train.get_context().get_world_rank()
    world = ray.train.get_context().get_world_size()
    x, y, _ = load_split(cfg["data"], "train")
    x, y = x[rank::world], y[rank::world]      # shard across workers
    model = ray.train.torch.prepare_model(build_model(cfg, len(class_names())))
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    xt = torch.from_numpy(x.transpose(0, 3, 1, 2)).float()
    yt = torch.from_numpy(y).long()
    for epoch in range(cfg["epochs"]):
        opt.zero_grad(); loss = F.cross_entropy(model(xt), yt); loss.backward(); opt.step()
        ray.train.report({"loss": float(loss)})

if HAVE_RAY:
    if not ray.is_initialized():
        ray.init(num_cpus=4, logging_level="ERROR", include_dashboard=False)
    trainer = TorchTrainer(
        train_func,
        train_loop_config={"data": str(DATA.resolve()), "lr": 1e-3, "epochs": 3,
                           "model": {"name": "tiny_cnn", "width": 16, "depth": 2}},
        scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
    )
    result = trainer.fit()
    m = result.metrics or {}
    loss = m.get("loss")
    if loss is None and getattr(result, "metrics_dataframe", None) is not None \
            and "loss" in result.metrics_dataframe:
        loss = float(result.metrics_dataframe["loss"].iloc[-1])
    print("Ray Train ran across 2 data-parallel workers.")
    print("final loss:", round(loss, 4) if loss is not None else "(metrics not surfaced)")
else:
    print("(install ray[train] to run this)")

(TrainController pid=7962) Requesting resources: {'CPU': 1} * 2


(TrainController pid=7962) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(RayTrainWorker pid=8062) [W525 16:13:51.133681786 socket.cpp:764] [c10d] The client socket cannot be initialized to connect to [::ffff:192.0.2.2]:42213 (errno: 97 - Address family not supported by protocol).
(RayTrainWorker pid=8064) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=7962) Started training worker group of size 2: 
(TrainController pid=7962) - (ip=192.0.2.2, pid=8064) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=7962) - (ip=192.0.2.2, pid=8062) world_rank=1, local_rank=1, node_rank=0
(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


...

(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=8064) /tmp/ipykernel_7646/1984160108.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
(RayTrainWorker pid=8064) Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
(RayTrainWorker pid=8064) Reporting training result 3: TrainingReport(checkpoint=None, metrics={'loss': 0.9858428835868835}, validation=False) [repeated 5x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


(PlacementGroupCleaner pid=8054) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


Ray Train ran across 2 data-parallel workers.
final loss: (metrics not surfaced)


Two workers each trained on their shard with synchronized gradients
(`prepare_model` wraps DistributedDataParallel). The evaluator still scores the
**full, unsharded** test split on the driver — sharding evaluation would quietly
change the worst-group metric.